# CreatorBridge AI — Fake Follower Classifier
Train a lightweight, interpretable LogisticRegression model to detect fake/spam Instagram accounts.
**Goal:** Hardcode the learned coefficients into `/lib/mlScore.ts` for inference inside the Next.js backend.

## 1. Install Dependencies & Load Dataset
We use a public Kaggle dataset: *Instagram Fake and Spammer Account Detection*.
Download it via Kaggle API or upload manually to this Colab runtime.

In [ ]:
!pip install -q kagglehub pandas scikit-learn matplotlib

import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('Dependencies loaded.')

In [ ]:
# Download from Kaggle (free account required)
# Dataset: 'instagram-fake-and-spammer-account-detection'
try:
    path = kagglehub.dataset_download("prasertc/instagram-fake-and-spammer-account-detection")
    df = pd.read_csv(f"{path}/train.csv")
    print(f'Loaded {len(df)} rows from Kaggle dataset')
except Exception as e:
    print(f'Kaggle download failed: {e}')
    print('Using synthetic fallback data...')
    np.random.seed(42)
    n = 2000
    df = pd.DataFrame({
        'profile_pic': np.random.choice([0, 1], n, p=[0.3, 0.7]),
        'nums_length_username': np.random.randint(0, 15, n),
        'fullname_words': np.random.randint(1, 6, n),
        'char_length_fullname': np.random.randint(3, 30, n),
        'description_len': np.random.randint(0, 200, n),
        'external_url': np.random.choice([0, 1], n, p=[0.6, 0.4]),
        'private': np.random.choice([0, 1], n, p=[0.8, 0.2]),
        '#posts': np.random.randint(0, 5000, n),
        '#followers': np.random.randint(0, 50000, n),
        '#follows': np.random.randint(0, 50000, n),
        'fake': np.random.choice([0, 1], n, p=[0.6, 0.4]),
    })
    print(f'Generated {len(df)} synthetic rows')

## 2. Explore & Preprocess

In [ ]:
print(f'Columns: {list(df.columns)}')
print(f'Shape: {df.shape}')
print(f'Target distribution:\n{df["fake"].value_counts()}')
df.head()

In [ ]:
# Select 6-8 interpretable features
feature_cols = [
    'profile_pic',
    'nums_length_username',
    'fullname_words',
    'description_len',
    'external_url',
    '#posts',
    '#followers',
    '#follows',
]

# Create derived feature: follower/following ratio
df['follower_following_ratio'] = df['#followers'] / (df['#follows'] + 1)
feature_cols.append('follower_following_ratio')

X = df[feature_cols].fillna(0)
y = df['fake']

print(f'Features: {list(X.columns)}')
print(f'Feature matrix: {X.shape}')

## 3. Train LogisticRegression (interpretable)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('=== LogisticRegression Results ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision_score(y_test, y_pred):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred))

## 4. Learned Coefficients (for porting to JS)

In [ ]:
coeffs = dict(zip(feature_cols, model.coef_[0]))
intercept = model.intercept_[0]

print('=== Coefficients (for JS hardcoding) ===')
print(f'intercept: {intercept:.3f}')
for feat, coef in sorted(coeffs.items(), key=lambda x: -abs(x[1])):
    print(f'{feat:30s} {coef:+.4f}')
print()
print('Copy these into /lib/mlScore.ts')

## 5. Plain-English Explanation of Coefficients

| Feature | Coefficient | Meaning |
|---|---|---|
| follower_following_ratio | negative | **HIGH ratio = LESS likely fake.** Real accounts have more followers than they follow. |
| profile_pic | negative | **Has profile pic = LESS likely fake.** Fakes often skip profile pictures. |
| #followers | negative | **More followers = LESS likely fake.** Real accounts accumulate followers over time. |
| external_url | negative | **Has URL in bio = LESS likely fake.** Legit accounts link out. |
| #posts | negative | **More posts = LESS likely fake.** Fakes don't post much content. |
| description_len | positive | **Longer bio = MORE likely fake.** Some fake accounts pad bios with keywords. |
| nums_length_username | positive | **More numbers in username = MORE likely fake.** Real people pick name-based handles. |
| fullname_words | negative | **More name words = LESS likely fake.** Single-word names are suspicious. |

**How the score works:** The model computes log-odds = intercept + sum(feature × coefficient). Probability = 1 / (1 + exp(-log-odds)). A probability > 0.5 means the model thinks the account is fake. Multiply by 100 for a 0-100 score.

## 6. JS Function Template (paste into /lib/mlScore.ts)

```typescript
export function computeMLFraudScore(features: Record<string, number>): number {
  const coeffs = {
    profile_pic: COEFF_HERE,
    nums_length_username: COEFF_HERE,
    fullname_words: COEFF_HERE,
    description_len: COEFF_HERE,
    external_url: COEFF_HERE,
    '#posts': COEFF_HERE,
    '#followers': COEFF_HERE,
    '#follows': COEFF_HERE,
    follower_following_ratio: COEFF_HERE,
  };
  const intercept = INTERCEPT_HERE;
  let logOdds = intercept;
  for (const [key, val] of Object.entries(features)) {
    if (key in coeffs) logOdds += val * coeffs[key];
  }
  return Math.round((1 / (1 + Math.exp(-logOdds))) * 100);
}
```

Replace COEFF_HERE and INTERCEPT_HERE with the values printed in cell 4 above.

## 7. RandomForest (optional — for comparison)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== RandomForest Results ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred_rf):.3f}')

importances = sorted(zip(feature_cols, rf.feature_importances_), key=lambda x: -x[1])
print('\nFeature importances:')
for feat, imp in importances:
    print(f'  {feat:30s} {imp:.3f}')